# ProtoPNet – interpretable deep learning through prototypes

Na podstawie artykułu **Chen et al., 2019 – "This Looks Like That: Deep Learning for Interpretable Image Recognition"**.

Celem notebooka jest pokazanie, jak można zbudować model klasyfikacji obrazów,
który nie tylko osiąga wysoką dokładność, ale również potrafi wyjaśnić swoje decyzje
w sposób zrozumiały dla człowieka.

W kolejnych częściach notebooka zostaną przedstawione:

- opis i implementacja architektury
- trening modelu 
- wizualizacja prototypów
- interpretacja predykcji 

# Działanie i architektura modelu

ProtoPNet rozkłada obraz na fragmenty, wyszukuje prototypowe części obiektów i agreguje dowody dla każdej klasy. 

Model składa się z trzech głównych elementów:

### 1. Backbone CNN do ekstrakcji cech

Pierwsza część sieci to klasyczna sieć konwolucyjna (VGG, ResNet etc.), która przekształca obraz w embedding: image → CNN → embedding

### 2. Warstwa prototypów

Najważniejszym elementem modelu jest warstwa prototypów. Każdy prototyp reprezentuje charakterystyczny fragment obiektu dla danej klasy, np.:

- skrzydło ptaka
- dziób
- tłów

Najlpierw tworzymy prototypy, potem model oblicza podobieństwo między fragmentami obrazu a prototypami(embedding -> prototypes -> similarity). Im większe podobieństwo, tym większa aktywacja prototypu.

### 3. Warstwa klasyfikacji

Ostatnia warstwa modelu agreguje podobieństwa do prototypów i na ich podstawie podejmuje decyzję klasyfikacyjną: similarity → linear layer → prediction. Każdy prototyp dostarcza dowodów na rzecz danej klasy,a końcowa predykcja jest sumą tych dowodów. 

# Interpretacja predykcji

Największą zaletą ProtoPNet jest możliwość bezpośredniej interpretacji decyzji modelu na podstawie prototypów. Dla każdej predykcji możemy zobaczyć:

- który fragment obrazu aktywował prototyp
- który prototyp został użyty
- z jakiej klasy pochodzi ten prototyp


Dzięki temu użytkownik może łatwo zrozumieć, na jakich cechach obrazu model opiera swoją decyzję.

Schmat przedstawiający działanie modelu: 

<div style="text-align:center">

<img src="images/protopnet.png" width="1000">

<p style="font-size:14px">
Źródło:<a href="https://arxiv.org/pdf/1806.10574">
This Looks Like That: Deep Learning for Interpretable Image Recognition</a>
</p>

</div>

# Implementacja ProtoPNet

Model składa się z trzech głównych elementów:

## 1. **Backbone CNN** – do ekstrakcji reprezentacji obrazu


In [43]:
import torch
import torch.nn as nn
import torchvision.models as models

def get_backbone():
    
    backbone = models.resnet18(pretrained=True)
    
    # usuwamy ostatnią warstwę klasyfikacyjną, aby otrzymać reprezentacje
    backbone = nn.Sequential(*list(backbone.children())[:-2])
    
    return backbone

## 2. **Prototype Layer** – do porównania fragmentów obrazu z prototypami

In [44]:
class PrototypeLayer(nn.Module):
    
    def __init__(self, num_prototypes, prototype_dim):
        super().__init__()
        
        self.num_prototypes = num_prototypes
        
        # prototypy są parametrami modelu
        self.prototypes = nn.Parameter(
            torch.rand(num_prototypes, prototype_dim, 1, 1)
        )
    
    def forward(self, x):
        # x = feature map z CNN
        # wymiary: B × C × H × W
        
        batch_size = x.shape[0]
        distances = []
        
        for p in self.prototypes:
            # obliczamy odległość między patchami a prototypem
            dist = torch.sum((x - p)**2, dim=1)
            distances.append(dist)
        
        distances = torch.stack(distances, dim=1)
        
        # wybieramy najbardziej podobny patch
        similarity = -torch.min(distances.view(batch_size,
                                               self.num_prototypes,
                                               -1), dim=2)[0]
        
        return similarity

## 3. **Linear Classification Layer** – agregacja dowodów z prototypów


In [45]:
class ProtoPNet(nn.Module):
    
    def __init__(self, num_classes, num_prototypes=20, prototype_dim=512):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_prototypes = num_prototypes
        
        self.backbone = get_backbone()
        self.prototype_layer = PrototypeLayer(
            num_prototypes,
            prototype_dim
        )
        self.classifier = nn.Linear(num_prototypes, num_classes)
        
        self.initialize_last_layer()
    
    def initialize_last_layer(self):
        # każda klasa dostaje równą liczbę prototypów
        prototypes_per_class = self.num_prototypes // self.num_classes
        weight = torch.full((self.num_classes, self.num_prototypes), -0.5)
        for k in range(self.num_classes):
            start = k * prototypes_per_class
            end = (k + 1) * prototypes_per_class
            weight[k, start:end] = 1.0
        
        with torch.no_grad():
            self.classifier.weight.copy_(weight)
    
    def forward(self, x):
        features = self.backbone(x)
        similarity = self.prototype_layer(features)
        logits = self.classifier(similarity)
        return logits

## Przygotowanie datasetu i pipeline treningowego

W pracy ProtoPNet model trenowany był na datasetach
do szczegółowej klasyfikacji obrazów (fine-grained recognition), takich jak:

- **CUB-200-2011** – dataset ptaków
- **Stanford Cars** – klasyfikacja modeli samochodów

Natomiast, w eksperymentach autorów, najczęściej wykorzystywany jest
dataset **CUB-200-2011**, zawierający 11 000 obrazów praków z 200 klas (gatunków).

Uwaga: dataset zawiera wyłącznie etykiety klas, bez oznaczeń części obiektów. ProtoPNet sam uczy się tych prototypowych fragmentów obrazów podczas treningu.  

In [18]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

image_size = 224

train_transforms = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomCrop(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # normalizacja dla wybranego przez nas backbone - resnet18
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transforms = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [19]:
import os
import tarfile
import urllib.request

url = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"
dataset_path = "dataset/CUB_200_2011.tgz"
extract_path = "dataset"

os.makedirs("dataset", exist_ok=True)

if not os.path.exists(dataset_path):
    print("Downloading CUB-200-2011...")
    urllib.request.urlretrieve(url, dataset_path)

print("Extracting dataset...")
with tarfile.open(dataset_path, "r:gz") as tar:
    tar.extractall(path=extract_path)

print("Dataset ready.")

Extracting dataset...


/tmp/ipykernel_45762/3765397061.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Dataset ready.


In [20]:
import pandas as pd
import shutil

base_path = "dataset/CUB_200_2011"

images = pd.read_csv(
    f"{base_path}/images.txt",
    sep=" ",
    names=["img_id","filepath"]
)

labels = pd.read_csv(
    f"{base_path}/image_class_labels.txt",
    sep=" ",
    names=["img_id","target"]
)

split = pd.read_csv(
    f"{base_path}/train_test_split.txt",
    sep=" ",
    names=["img_id","is_train"]
)

data = images.merge(labels, on="img_id")
data = data.merge(split, on="img_id")

train_root = "dataset/train"
test_root = "dataset/test"

os.makedirs(train_root, exist_ok=True)
os.makedirs(test_root, exist_ok=True)

for _, row in data.iterrows():
    src = f"{base_path}/images/{row.filepath}"
    class_name = row.filepath.split("/")[0]
    if row.is_train == 1:
        dst_dir = os.path.join(train_root, class_name)
    else:
        dst_dir = os.path.join(test_root, class_name)
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, os.path.basename(row.filepath))
    if not os.path.exists(dst):
        shutil.copy(src, dst)

print("ImageFolder structure created.")



ImageFolder structure created.


In [21]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(
    root="dataset/train",
    transform=train_transforms
)

test_dataset = datasets.ImageFolder(
    root="dataset/test",
    transform=test_transforms
)

print("Number of classes:", len(train_dataset.classes))
print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Number of classes: 200
Train samples: 5994
Test samples: 5794


In [35]:
from torch.utils.data import DataLoader

batch_size = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=1
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1
)

### Funkcja straty 

Całkowita funkcja straty modelu ProtoPNet składa się z trzech składników:

$$
L = L_{cls} + \lambda_1 L_{cluster} + \lambda_2 L_{separation}
$$

gdzie:

- $L_{cls}$ – funkcja straty klasyfikacji (cross-entropy),
- $L_{cluster}$ – wymusza bliskość reprezentacji obrazów do prototypów tej samej klasy,
- $L_{separation}$ – wymusza oddalenie reprezentacji obrazów od prototypów innych klas,
- $\lambda_1, \lambda_2$ – współczynniki regularyzacji kontrolujące wpływ poszczególnych składników.

---

#### Classification loss

Funkcja klasyfikacji jest standardową funkcją **cross-entropy**:

$$
L_{cls} =
- \sum_{i=1}^{N}
\log
\frac{e^{f_{y_i}(x_i)}}
{\sum_{c=1}^{C} e^{f_c(x_i)}}
$$

gdzie:

- $x_i$ – obraz wejściowy,
- $y_i$ – prawdziwa etykieta klasy,
- $f_c(x_i)$ – logit modelu dla klasy $c$.

---

#### Cluster loss

Cluster loss zachęca, aby reprezentacja obrazu była **blisko prototypów swojej klasy**:

$$
L_{cluster} =
\frac{1}{N}
\sum_{i=1}^{N}
\min_{p_j \in P_{y_i}}
\| z_i - p_j \|_2^2
$$

gdzie:

- $z_i$ – reprezentacja obrazu w przestrzeni cech,
- $p_j$ – prototyp,
- $P_{y_i}$ – zbiór prototypów przypisanych do klasy $y_i$.

---

#### Separation loss

Separation loss wymusza, aby reprezentacja obrazu była **daleko od prototypów innych klas**:

$$
L_{separation} =
-
\frac{1}{N}
\sum_{i=1}^{N}
\min_{p_j \notin P_{y_i}}
\| z_i - p_j \|_2^2
$$

---

Dzięki tym trzem składnikom model jednocześnie:

- uczy się klasyfikować obrazy
- organizuje przestrzeń cech tak, aby obrazy były blisko prototypów swojej klasy a jednocześnie daleko od prototypów innych klas

In [ ]:
def cluster_loss(similarity, labels):
    batch_size = similarity.shape[0]
    loss = 0
    for i in range(batch_size):
        loss += -torch.max(similarity[i])
    return loss / batch_size

def separation_loss(similarity, labels):
    batch_size = similarity.shape[0]
    loss = 0
    for i in range(batch_size):
        loss += torch.mean(similarity[i])

    return loss / batch_size

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(train_dataset.classes)

model = ProtoPNet(
    num_classes=num_classes,
    num_prototypes=100,
    prototype_dim=256
).to(device)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 4.69 MiB is free. Including non-PyTorch memory, this process has 3.61 GiB memory in use. Of the allocated memory 3.38 GiB is allocated by PyTorch, and 148.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch.optim as optim

criterion = torch.nn.CrossEntropyLoss()

# -------------------------
# Funkcja treningowa
# -------------------------
def train_epoch(model, loader, optimizer):
    model.train()
    
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        features = model.backbone(images)
        similarity = model.prototype_layer(features)
        logits = model.classifier(similarity)

        # Straty
        loss_cls = criterion(logits, labels)
        loss_cluster = cluster_loss(similarity, labels)
        loss_sep = separation_loss(similarity, labels)

        loss = loss_cls + 0.8 * loss_cluster + 0.1 * loss_sep

        # Backward + update
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    return total_loss / len(loader), acc

In [ ]:
# Zamrażamy classifier
for param in model.classifier.parameters():
    param.requires_grad = False

# Optymalizator dla backbone + prototype layer
optimizer_step1 = optim.Adam(
    list(model.backbone.parameters()) + list(model.prototype_layer.parameters()),
    lr=1e-4
)

In [37]:
# Odmrażamy classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# Optymalizator dla całego modelu
optimizer_step2 = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [39]:
num_epochs_step1 = 5
num_epochs_step2 = 5

# Step 1
for epoch in range(num_epochs_step1):
    loss, acc = train_epoch(model, train_loader, optimizer_step1)
    print(f"[Step 1] Epoch {epoch+1}/{num_epochs_step1} - Loss: {loss:.4f}, Acc: {acc:.4f}")

# Step 2
for epoch in range(num_epochs_step2):
    loss, acc = train_epoch(model, train_loader, optimizer_step2)
    print(f"[Step 2] Epoch {epoch+1}/{num_epochs_step2} - Loss: {loss:.4f}, Acc: {acc:.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 4.69 MiB is free. Including non-PyTorch memory, this process has 3.61 GiB memory in use. Of the allocated memory 3.37 GiB is allocated by PyTorch, and 152.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

In [40]:
plt.figure(figsize=(8,5))
plt.plot(train_accuracies, label="Train accuracy")
plt.plot(test_accuracies, label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()
plt.show()

NameError: name 'plt' is not defined

In [41]:
import torchvision
import numpy as np

model.eval()

images, labels = next(iter(test_loader))
images = images.to(device)

with torch.no_grad():
    logits = model(images)
    preds = torch.argmax(logits, dim=1)

images = images.cpu()

fig, axes = plt.subplots(3, 3, figsize=(8,8))

for i, ax in enumerate(axes.flat):

    img = images[i].permute(1,2,0).numpy()

    ax.imshow((img - img.min()) / (img.max() - img.min()))
    
    ax.set_title(
        f"Pred: {train_dataset.classes[preds[i]]}\n"
        f"True: {train_dataset.classes[labels[i]]}",
        fontsize=8
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 4.69 MiB is free. Including non-PyTorch memory, this process has 3.61 GiB memory in use. Of the allocated memory 3.37 GiB is allocated by PyTorch, and 152.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [42]:
def visualize_prototype_activation(image):

    model.eval()

    with torch.no_grad():

        image = image.unsqueeze(0).to(device)

        features = model.backbone(image)

        similarity = model.prototype_layer(features)

        prototype_id = torch.argmax(similarity)

    return prototype_id.item()

In [ ]:
## Interpretacja wyników

Analiza wyników pozwala zauważyć kilka ważnych właściwości ProtoPNet:

- model osiąga konkurencyjną **dokładność klasyfikacji**
- jednocześnie zapewnia **interpretowalne decyzje**

Dla każdej predykcji możemy wskazać:

1. fragment obrazu,
2. prototyp,
3. klasę, której prototyp dostarcza dowodu.

Wyjaśnienie modelu ma więc formę:


# Zadanie 1 (obowiązkowe)
Wykorzystując podany kod, wytrenuj go na prostym datasecie, takim jak MNIST lub CIFAR10 i zwizualizuj wyniki. Czy osiągnięte wyniki są Twoim zdaniem akceptowalne, czy nie? Z czego może wynikać taki poziom skuteczności modelu w kontekście wybranego zbioru danych?